# Calendar Dimension Table

Builds the standard calendar dimension `caresync_adb.silver.calendar` covering 1940-01-01 to 2030-12-31, one row per day.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS caresync_adb.silver;

CREATE OR REPLACE TABLE caresync_adb.silver.calendar
COMMENT 'Standard calendar dimension, one row per day from 1940-01-01 to 2030-12-31'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS d
)
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT)              AS date_key,
  d                                                    AS date,
  year(d)                                              AS year,
  quarter(d)                                           AS quarter,
  concat('Q', quarter(d))                              AS quarter_name,
  concat(year(d), '-Q', quarter(d))                    AS year_quarter,
  CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END            AS half_year,
  month(d)                                             AS month,
  date_format(d, 'MMMM')                               AS month_name,
  date_format(d, 'MMM')                                AS month_short_name,
  date_format(d, 'yyyy-MM')                            AS year_month,
  CAST(date_format(d, 'yyyyMM') AS INT)                AS year_month_key,
  day(d)                                               AS day,
  dayofyear(d)                                         AS day_of_year,
  weekday(d) + 1                                       AS day_of_week,        -- ISO: 1=Monday ... 7=Sunday
  date_format(d, 'EEEE')                               AS day_name,
  date_format(d, 'EEE')                                AS day_short_name,
  weekofyear(d)                                        AS week_of_year,       -- ISO week number
  extract(YEAROFWEEK FROM d)                           AS iso_year,
  CAST(ceil(day(d) / 7.0) AS INT)                      AS week_of_month,
  date_trunc('WEEK', d)::DATE                          AS week_start_date,    -- Monday
  date_add(date_trunc('WEEK', d)::DATE, 6)             AS week_end_date,      -- Sunday
  trunc(d, 'MM')                                       AS month_start_date,
  last_day(d)                                          AS month_end_date,
  day(last_day(d))                                     AS days_in_month,
  date_trunc('QUARTER', d)::DATE                       AS quarter_start_date,
  date_sub(add_months(date_trunc('QUARTER', d)::DATE, 3), 1) AS quarter_end_date,
  trunc(d, 'YYYY')                                     AS year_start_date,
  make_date(year(d), 12, 31)                           AS year_end_date,
  weekday(d) >= 5                                      AS is_weekend,
  weekday(d) < 5                                       AS is_weekday,
  d = trunc(d, 'MM')                                   AS is_month_start,
  d = last_day(d)                                      AS is_month_end,
  d = date_trunc('QUARTER', d)::DATE                   AS is_quarter_start,
  d = date_sub(add_months(date_trunc('QUARTER', d)::DATE, 3), 1) AS is_quarter_end,
  (month(d) = 1 AND day(d) = 1)                        AS is_year_start,
  (month(d) = 12 AND day(d) = 31)                      AS is_year_end,
  ((year(d) % 4 = 0 AND year(d) % 100 <> 0) OR year(d) % 400 = 0) AS is_leap_year
FROM dates
ORDER BY d;

In [0]:
%sql
SELECT
  count(*)                    AS total_rows,
  count(DISTINCT date_key)    AS distinct_keys,
  min(date)                   AS min_date,
  max(date)                   AS max_date
FROM caresync_adb.silver.calendar;

In [0]:
%sql
SELECT * FROM caresync_adb.silver.calendar 